<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/python/notebooks/c2_l3.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C2-L3 · NumPy vectorizado
Retornos, SMA y señales sin bucles. Comparamos velocidad bucle vs vector.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path

URL = 'https://raw.githubusercontent.com/Emelecto/QuantLab/main/web/content/cursos/python/data/c2_l3.csv'
try:
    df = pd.read_csv(URL, parse_dates=['date'])
    print('Fuente: URL (Colab)')
except Exception as e:
    print('Sin red, uso fallback local:', e)
    for cand in [Path('../data/c2_l3.csv'), Path('data/c2_l3.csv'), Path('c2_l3.csv')]:
        if cand.exists():
            df = pd.read_csv(cand, parse_dates=['date']); break
    print('Fuente: local')
print(df.shape)

In [ ]:
import numpy as np, time
c = df['close'].to_numpy()
# Bucle
t0 = time.perf_counter()
rets_loop = np.array([(c[i]-c[i-1])/c[i-1] for i in range(1, len(c))])
t_loop = time.perf_counter() - t0
# Vector
t0 = time.perf_counter()
rets_vec = np.diff(c) / c[:-1]
t_vec = time.perf_counter() - t0
print(f'bucle={t_loop*1e6:.1f} µs  vector={t_vec*1e6:.1f} µs')
print(f'máx dif={np.max(np.abs(rets_loop - rets_vec)):.2e}')

In [ ]:
df = df.sort_values('date').reset_index(drop=True)
df['ret_simple'] = df['close'].pct_change()
df['ret_log'] = np.log(df['close'] / df['close'].shift(1))
df['sma20'] = df['close'].rolling(20).mean()
df['senal'] = np.where(df['close'] > df['sma20'], 1, -1)
print(df[['date','close','sma20','senal']].tail(8).to_string(index=False))
cambios = int((df['senal'].dropna() != df['senal'].dropna().shift(1)).sum())
print(f'\ncambios_de_señal={cambios}')

In [ ]:
# Chequeo automático
assert np.allclose(rets_loop, rets_vec), 'bucle y vector deben coincidir'
assert df['sma20'].iloc[19] == df['close'].iloc[:20].mean(), 'SMA20 = media de los 20 primeros'
assert set(df['senal'].dropna().unique()) <= {1, -1}
print('OK: vectorización verificada')